# Retraining z użyciem klienta ML App

Ten sam scenariusz integracyjny co w przykładzie surowego API, ale bez ręcznego składania requestów i rozwiązywania identyfikatorów. Klient strumieniuje plik, wybiera najnowszą opublikowaną wersję pipeline'u i może czekać na wynik.

In [ ]:
# JEDYNE DANE SCENARIUSZA
BUSINESS_CASE_NAME = "Estates Sell Prices"
DATASET_NAME = "sale-prices"
PIPELINE_NAME = "Estates Sell Prices - AutoFEML"
INPUT_FILE = None  # np. r"C:\\data\\sale-prices.csv"; None tworzy demo 30k rekordów

## Klient i uwierzytelnienie

Zainstaluj klienta z katalogu repozytorium przez `pip install -e .`. W automatyzacji ustaw `ML_APP_API_URL` i `ML_APP_ACCESS_TOKEN` w secret managerze. Lokalnie notebook bezpiecznie poprosi o poświadczenia, jeśli tokenu nie ma.

In [ ]:
import getpass
import os
from pathlib import Path

from ml_app_client import MLAppClient

client = MLAppClient.from_env()
if not os.getenv("ML_APP_ACCESS_TOKEN", "").strip():
    client.login(input("Login: "), getpass.getpass("Hasło: "))

## Plik wejściowy

Generator służy wyłącznie demonstracji i nie wykonuje requestów do platformy.

In [ ]:
if INPUT_FILE:
    training_file = Path(INPUT_FILE)
else:
    import sys
    helpers = next(path for path in [Path.cwd(), Path.cwd() / "examples" / "API-usage"] if (path / "api_example_helpers.py").is_file())
    sys.path.insert(0, str(helpers))
    from api_example_helpers import build_training_file
    training_file = build_training_file()
training_file

## Upload kolejnej wersji datasetu

Klient odnajduje dataset przypięty do Business Case i wysyła plik strumieniowo jako jego nową, niemutowalną wersję.

In [ ]:
uploaded = client.upload_dataset_version(
    training_file,
    business_case_name=BUSINESS_CASE_NAME,
    dataset_name=DATASET_NAME,
)
print(f"Wgrano {uploaded.name} v{uploaded.version_number} ({uploaded.row_count:,} wierszy)")

## Start pipeline'u

Klient wybiera najnowszą opublikowaną wersję. Pipeline z wejściem `latest` zostanie przez backend związany z wersją wysłaną powyżej.

In [ ]:
run = client.run_pipeline_by_name(
    business_case_name=BUSINESS_CASE_NAME,
    pipeline_name=PIPELINE_NAME,
)
print(f"Uruchomiono run_id={run.id}, status={run.status}")

## Oczekiwanie na wynik

Polling pobiera wyłącznie ograniczone metadane runu. Pełny dataset nie trafia do notebooka. Timeout nie anuluje zadania na platformie.

In [ ]:
finished = client.wait_for_pipeline_run(
    run,
    timeout=3600,
    on_update=lambda current: print(f"status={current.status}, przetworzono={current.processed_row_count}"),
)
print(f"Gotowe: {finished.processed_row_count:,} przetworzonych wierszy")